## 1. Imports and settings

In [4]:
#numpy pandas scipy scikit-learn matplotlib folium joblib
import json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import folium
import joblib
import sklearn

from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import norm

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             log_loss, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

K_CAND = 10          # candidate roads kept per GPS observation
TRAIN_STRIDE = 5     # use every 5th GPS observation of a training trip for model fitting (speed)
FLAG_DIST_M = 50     # trip is flagged if its median GPS-to-reference-route distance exceeds this
CV_FOLDS = 5

# Only the three FastAPI handoff artifacts are saved here.
OUT = Path("pathsnap_output-1")
OUT.mkdir(exist_ok=True)
FEATURES = ["distance_m", "midpoint_distance_m", "segment_length_m", "speed_mps",
            "direction_difference", "distance_rank", "relative_distance_m"]

## 2. Locate the dataset
Keep the dataset ZIP next to this notebook (any `*.zip`) or set `ZIP_PATH`. If the data is already extracted into `pathsnap_data/`, the ZIP is not needed.

In [5]:
DATA_DIR = Path("Student folder")  # Change this to your extracted dataset folder if needed

def find_dirs(root):
    dirs = [p for p in root.rglob("*") if p.is_dir() and "__MACOSX" not in p.parts]

    train = [p for p in dirs if p.name.lower() == "train"]
    test = [p for p in dirs if "test" in p.name.lower() and "evaluator" not in p.name.lower()]

    return (train[0] if train else None), (test[0] if test else None)


TRAIN_DIR, STUDENT_TEST_DIR = find_dirs(DATA_DIR)

assert TRAIN_DIR is not None, "No 'train' folder found."
assert STUDENT_TEST_DIR is not None, "No student-test folder found."

print("Training folder    :", TRAIN_DIR)
print("Student-test folder:", STUDENT_TEST_DIR)


Training folder    : Student folder/train
Student-test folder: Student folder/student_test


## 6. Process every trip (EDA statistics + candidates + labels)
One pass per partition. For each trip we record data-quality, time-gap, extent, road-network connectivity (isolated nodes, connected components), route structure and how far the GPS trace is from the reference route.

In [ ]:
def trip_eda(data, geom, ref):
    tr, arcs, P = data["track"], geom["arcs"], geom["P"]
    gaps = tr["time"].diff().dropna()
    n_nodes = len(data["nodes"])
    adj = coo_matrix((np.ones(len(arcs)), (arcs[:, 0], arcs[:, 1])), shape=(n_nodes, n_nodes))
    n_comp, labels = connected_components(adj, directed=False)
    used = np.zeros(n_nodes, bool); used[arcs.ravel()] = True
    route = np.array(data["route"])
    route_ok = float(np.mean(arcs[route[:-1], 1] == arcs[route[1:], 0])) if len(route) > 1 else np.nan
    row = {"trip_id": data["trip_id"], "gps_points": len(tr), **data["quality"],
           "median_gap_s": float(gaps.median()), "max_gap_s": float(gaps.max()),
           "share_1s_steps": float((gaps == 1).mean()), "gaps_over_10s": int((gaps > 10).sum()),
           "extent_x_km": float(np.ptp(P[:, 0]) / 1000), "extent_y_km": float(np.ptp(P[:, 1]) / 1000),
           "road_nodes": n_nodes, "road_arcs": len(arcs), "isolated_nodes": int((~used).sum()),
           "components": int(n_comp), "largest_component_share": float(np.bincount(labels).max() / n_nodes),
           "route_arcs": len(route), "route_connected_share": route_ok,
           "median_ref_dist_m": float(ref["ref_distance_m"].median())}
    row["flagged"] = row["median_ref_dist_m"] > FLAG_DIST_M
    return row

def process_partition(folder, name):
    frames, rows, store = [], [], {}
    t0 = time.time()
    for td in trip_folders(folder):
        data = read_trip(td)
        geom = trip_geometry(data)
        ref = reference_arcs(data, geom)
        frames.append(add_labels(generate_candidates(data, geom), ref))
        rows.append(trip_eda(data, geom, ref))
        store[data["trip_id"]] = {"arcs": geom["arcs"], "route": data["route"],
                                  "ref": ref.set_index("obs_index")["ref_arc"]}
    print(f"{name}: {len(rows)} trips processed in {time.time() - t0:.0f}s")
    return pd.concat(frames, ignore_index=True), pd.DataFrame(rows), store

train_all, eda_train, store_train = process_partition(TRAIN_DIR, "train")
test_all, eda_test, store_test = process_partition(STUDENT_TEST_DIR, "student_test")

### 6.1 Data-quality summary

In [ ]:
for name, e in [("TRAIN", eda_train), ("STUDENT-TEST", eda_test)]:
    print(f"===== {name}: {len(e)} trips =====")
    cols = ["gps_points", "missing_values", "rows_removed", "unsorted_steps", "duplicate_times", "max_gap_s",
            "share_1s_steps", "gaps_over_10s", "road_nodes", "road_arcs", "isolated_nodes", "components",
            "route_arcs", "route_connected_share", "median_ref_dist_m"]
    display(e[cols].describe().T[["min", "50%", "max"]].round(3))
    print("Total missing values:", int(e["missing_values"].sum()),
          "| total rows removed by cleaning:", int(e["rows_removed"].sum()))
    print("Flagged trips (median GPS-to-reference-route distance >", FLAG_DIST_M, "m):",
          e.loc[e["flagged"], "trip_id"].tolist())

### 6.2 Exploration plots (GPS counts, time gaps, trajectories, road network)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(eda_train["gps_points"], bins=25); ax[0].set_title("GPS observations per training trip")
ax[0].set_xlabel("observations"); ax[0].set_ylabel("trips")
ax[1].hist(eda_train["max_gap_s"], bins=25, color="tab:orange"); ax[1].set_yscale("log")
ax[1].set_title("Largest time gap per training trip"); ax[1].set_xlabel("seconds")
ax[2].hist(eda_train["median_ref_dist_m"], bins=25, color="tab:green")
ax[2].axvline(FLAG_DIST_M, color="red", ls="--"); ax[2].set_title("Median GPS-to-route distance per trip")
ax[2].set_xlabel("metres")
plt.tight_layout(); plt.show()

# Trajectories of four training trips (longitude / latitude)
ids = list(eda_train["trip_id"][:4])
fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
for a, tid in zip(ax, ids):
    tr = read_trip(TRAIN_DIR / tid)["track"]
    a.plot(tr["lon"], tr["lat"], ".", ms=1.5); a.set_title(f"Trip {tid} ({len(tr)} pts)")
    a.set_xlabel("lon"); a.set_ylabel("lat")
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib.collections import LineCollection

def plot_network(trip_dir, ax, pad_m=300):
    d = read_trip(trip_dir); g = trip_geometry(d)
    P, S, E = g["P"], g["S"], g["E"]
    lo, hi = P.min(0) - pad_m, P.max(0) + pad_m
    inside = ((S > lo) & (S < hi)).all(1) | ((E > lo) & (E < hi)).all(1)
    segs = np.stack([S[inside], E[inside]], axis=1)
    ax.add_collection(LineCollection(segs, colors="lightgrey", linewidths=0.6))
    r = np.array(d["route"])
    ax.add_collection(LineCollection(np.stack([S[r], E[r]], axis=1), colors="green", linewidths=2, label="reference route"))
    ax.plot(P[:, 0], P[:, 1], "b.", ms=1.5, label="GPS")
    ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_aspect("equal")
    ax.set_title(f"Trip {d['trip_id']}: road network, GPS and reference route"); ax.legend(loc="upper right")

good = eda_train.loc[~eda_train["flagged"], "trip_id"].tolist()
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
plot_network(TRAIN_DIR / good[0], ax[0]); plot_network(TRAIN_DIR / good[min(3, len(good) - 1)], ax[1])
plt.tight_layout(); plt.show()

## 7. Training and student-test tables
Training rows come from the non-flagged training trips, using every `TRAIN_STRIDE`-th GPS observation (keeps the five models and Bayesian search fast; the sampled rows still cover every trip). The **student-test table keeps every observation**, so Top-1, candidate-miss and route continuity are computed on complete trips.

In [ ]:
flagged_train = set(eda_train.loc[eda_train["flagged"], "trip_id"])
train_tbl = train_all[~train_all["trip_id"].isin(flagged_train) & (train_all["obs_index"] % TRAIN_STRIDE == 0)].reset_index(drop=True)
test_tbl = test_all.reset_index(drop=True)

X_train, y_train, g_train = train_tbl[FEATURES], train_tbl["is_match"], train_tbl["trip_id"]
X_test, y_test = test_tbl[FEATURES], test_tbl["is_match"]

_, miss_train = top1_and_miss(train_all[~train_all["trip_id"].isin(flagged_train)], np.zeros((~train_all["trip_id"].isin(flagged_train)).sum()))
_, miss_test = top1_and_miss(test_tbl, np.zeros(len(test_tbl)))
print(f"Training rows: {len(train_tbl):,} from {g_train.nunique()} trips "
      f"({len(flagged_train)} flagged trips excluded) | positive rate {y_train.mean():.3f}")
print(f"Student-test rows: {len(test_tbl):,} from {test_tbl['trip_id'].nunique()} trips | positive rate {y_test.mean():.3f}")
print(f"Candidate miss rate  - training trips (flagged excluded): {miss_train:.4f} | student-test trips: {miss_test:.4f}")
assert set(train_tbl["trip_id"]).isdisjoint(set(test_tbl["trip_id"])), "trip leaked across partitions"
display(train_tbl[["trip_id", "obs_index", "arc_id"] + FEATURES + ["is_match"]].head())

## 8. Compare five models
All five output a match probability for every candidate road, so the HMM can use them later.

* **Primary selection metric: CV Macro-F1** (group 5-fold CV on the training trips, each trip kept in one fold). Macro-F1 is used because only ~10 % of candidates are positives, so accuracy alone is misleading. CV log-loss is the second criterion (probability quality matters because the HMM uses the probabilities).
* Scaling/imputation are inside the pipelines, so they are fitted on training folds only.
* The student-test set is **not** used to pick the final model.

In [ ]:
models = {
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
        LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    "K-Nearest Neighbours": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
        KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1)),
    "Decision Tree": make_pipeline(SimpleImputer(strategy="median"),
        DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, class_weight="balanced", random_state=RANDOM_STATE)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=100, max_depth=14, min_samples_leaf=5, class_weight="balanced",
                               n_jobs=-1, random_state=RANDOM_STATE)),
    "Hist Gradient Boosting": make_pipeline(SimpleImputer(strategy="median"),
        HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, max_depth=6, early_stopping=False,
                                       random_state=RANDOM_STATE)),
}
model_info = pd.DataFrame([
    ("Logistic Regression", "Linear / probabilistic", "C=1.0, L2, balanced class weight, standardised", "Log loss (cross-entropy)"),
    ("K-Nearest Neighbours", "Instance-based", "k=15, distance-weighted, standardised", "None (distance-weighted vote)"),
    ("Decision Tree", "Tree-based", "max_depth=10, min_samples_leaf=20, balanced", "Gini impurity"),
    ("Random Forest", "Ensemble (bagging)", "100 trees, max_depth=14, min_samples_leaf=5, balanced", "Gini impurity per tree"),
    ("Hist Gradient Boosting", "Boosting", "150 iters, lr=0.1, max_depth=6", "Log loss (gradient boosted)"),
], columns=["Model", "Algorithm family", "Key configuration", "Objective"]).set_index("Model")

cv = GroupKFold(n_splits=CV_FOLDS)
fitted, cv_rows, test_scores, test_rows = {}, [], {}, []
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, groups=g_train, cv=cv, n_jobs=1, return_train_score=True,
                         scoring={"f1": "f1_macro", "ll": "neg_log_loss"})
    t0 = time.time(); model.fit(X_train, y_train); fit_s = time.time() - t0
    t0 = time.time(); p = model.predict_proba(X_test)[:, 1]; inf_s = time.time() - t0
    fitted[name], test_scores[name] = model, p
    pred = (p >= 0.5).astype(int)
    top1, miss = top1_and_miss(test_tbl, p)
    cv_rows.append({"Model": name, "CV_Macro_F1": res["test_f1"].mean(), "CV_F1_std": res["test_f1"].std(),
                    "CV_Log_Loss": -res["test_ll"].mean(), "Train_Macro_F1": res["train_f1"].mean(),
                    "Fit_time_s": fit_s})
    test_rows.append({"Model": name, "Test_Accuracy": accuracy_score(y_test, pred),
                      "Test_Precision": precision_score(y_test, pred, zero_division=0),
                      "Test_Recall": recall_score(y_test, pred, zero_division=0),
                      "Test_Macro_F1": f1_score(y_test, pred, average="macro"),
                      "Test_Log_Loss": log_loss(y_test, np.clip(p, 1e-6, 1 - 1e-6)),
                      "Top1_Accuracy": top1, "Candidate_Miss_Rate": miss, "Inference_time_s": inf_s})
    print(f"{name:24s} CV Macro-F1 {cv_rows[-1]['CV_Macro_F1']:.4f} | fit {fit_s:.1f}s")

**Route continuity** (classifier-only) is the share of consecutive *different* arcs in the predicted route where the next arc starts at the previous arc's end node. Repeated arcs are collapsed first, because a vehicle stays on one arc for several 1 Hz observations.

In [ ]:
def route_continuity(seq, arcs):
    seq = [int(a) for i, a in enumerate(seq) if i == 0 or a != seq[i - 1]]
    if len(seq) < 2:
        return np.nan
    ok = sum(int(arcs[a][1]) == int(arcs[b][0]) for a, b in zip(seq[:-1], seq[1:]))
    return ok / (len(seq) - 1)

def arc_set_scores(pred, ref):
    p, r = set(map(int, pred)), set(map(int, ref))
    tp = len(p & r)
    prec = tp / len(p) if p else 0.0
    rec = tp / len(r) if r else 0.0
    return prec, rec, (2 * prec * rec / (prec + rec) if prec + rec else 0.0)

def classifier_only_routes(frame, scores):
    # best-scoring candidate for every GPS observation, returned per trip
    tmp = frame[["trip_id", "obs_index", "arc_id"]].copy(); tmp["score"] = np.asarray(scores)
    best = tmp.loc[tmp.groupby(["trip_id", "obs_index"])["score"].idxmax()].sort_values(["trip_id", "obs_index"])
    return {tid: part for tid, part in best.groupby("trip_id")}

def mean_continuity(frame, scores):
    routes = classifier_only_routes(frame, scores)
    return float(np.nanmean([route_continuity(r["arc_id"].tolist(), store_test[t]["arcs"]) for t, r in routes.items()]))

for row in test_rows:
    row["Route_Continuity"] = mean_continuity(test_tbl, test_scores[row["Model"]])

comparison = (model_info.reset_index()
              .merge(pd.DataFrame(cv_rows), on="Model").merge(pd.DataFrame(test_rows), on="Model").set_index("Model"))
table_cols = ["Algorithm family", "Key configuration", "CV_Macro_F1", "Top1_Accuracy", "Route_Continuity",
              "Test_Macro_F1", "Test_Log_Loss", "Inference_time_s"]
display(comparison[table_cols].round(4))
display(comparison[["Objective", "Train_Macro_F1", "CV_Macro_F1", "CV_F1_std", "Test_Macro_F1", "CV_Log_Loss",
                    "Test_Precision", "Test_Recall", "Test_Accuracy", "Candidate_Miss_Rate", "Fit_time_s"]].round(4))

Overfitting check: a large gap between `Train_Macro_F1` and `CV_Macro_F1` signals overfitting; low values on both signal underfitting. Confusion matrices below are candidate-level on the student-test trips (threshold 0.5).

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(20, 3.8))
for a, (name, p) in zip(ax, test_scores.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, (p >= 0.5).astype(int)), display_labels=["no match", "match"]).plot(ax=a, colorbar=False)
    a.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
comparison[["Train_Macro_F1", "CV_Macro_F1", "Test_Macro_F1"]].plot.bar(ax=ax[0], rot=25); ax[0].set_title("Macro-F1: train vs CV vs student-test")
comparison[["Top1_Accuracy", "Route_Continuity"]].plot.bar(ax=ax[1], rot=25); ax[1].set_title("Student-test Top-1 and route continuity (classifier only)")
comparison[["CV_Log_Loss", "Test_Log_Loss"]].plot.bar(ax=ax[2], rot=25); ax[2].set_title("Log loss")
plt.tight_layout(); plt.show()

## 9. Bayesian hyperparameter optimisation (Gaussian process + Expected Improvement)
Gradient boosting is tuned: it is a strong, fast, probability-based model that suits the HMM.

* **Search space:** learning rate (log scale 0.02–0.3), iterations 50–300, tree depth 3–12, minimum samples per leaf 10–200, L2 regularisation 0–2.
* **Objective:** maximise mean **CV Macro-F1** over `GroupKFold` on the training trips (same folds as section 8).
* **Method:** 6 random initial trials, then each new trial maximises Expected Improvement under a Gaussian-process surrogate (Matérn kernel) fitted to all previous trials. Every trial is recorded.
* The student-test set is not used in the search.

In [ ]:
SPACE = {  # name: (low, high, kind)
    "learning_rate": (0.02, 0.3, "log"), "max_iter": (50, 300, "int"), "max_depth": (3, 12, "int"),
    "min_samples_leaf": (10, 200, "int"), "l2_regularization": (0.0, 2.0, "float")}
N_INITIAL, N_BAYES = 6, 14
names = list(SPACE)

def decode(u):
    params = {}
    for x, name in zip(u, names):
        lo, hi, kind = SPACE[name]
        if kind == "log":
            params[name] = float(np.exp(np.log(lo) + x * (np.log(hi) - np.log(lo))))
        elif kind == "int":
            params[name] = int(round(lo + x * (hi - lo)))
        else:
            params[name] = float(lo + x * (hi - lo))
    return params

def make_hgb(params):
    return make_pipeline(SimpleImputer(strategy="median"),
                         HistGradientBoostingClassifier(early_stopping=False, random_state=RANDOM_STATE, **params))

def objective(u):
    t0 = time.time()
    params = decode(u)
    res = cross_validate(make_hgb(params), X_train, y_train, groups=g_train, cv=cv, scoring="f1_macro", n_jobs=1)
    return res["test_score"].mean(), res["test_score"].std(), time.time() - t0

def expected_improvement(gp, cand, best):
    mu, sd = gp.predict(cand, return_std=True)
    sd = np.maximum(sd, 1e-9)
    z = (mu - best) / sd
    return (mu - best) * norm.cdf(z) + sd * norm.pdf(z)

rng = np.random.RandomState(RANDOM_STATE)
U, scores, trials = [], [], []
t_bo = time.time()
for trial in range(N_INITIAL + N_BAYES):
    if trial < N_INITIAL:
        u = rng.rand(len(names)); how = "random"
    else:
        gp = GaussianProcessRegressor(kernel=Matern(nu=2.5) + WhiteKernel(1e-3), normalize_y=True,
                                      n_restarts_optimizer=3, random_state=RANDOM_STATE)
        gp.fit(np.array(U), np.array(scores))
        pool = rng.rand(3000, len(names))
        u = pool[np.argmax(expected_improvement(gp, pool, max(scores)))]; how = "bayesian (EI)"
    score, std, secs = objective(u)
    U.append(u); scores.append(score)
    trials.append({"trial": trial + 1, "method": how, "cv_macro_f1": score, "cv_std": std, "seconds": secs, **decode(u)})
    print(f"trial {trial + 1:2d} [{how:13s}] CV Macro-F1 {score:.4f}  {decode(u)}")

bo_trials = pd.DataFrame(trials)
best_params = decode(U[int(np.argmax(scores))])
bo_seconds = time.time() - t_bo
print("\nBest CV Macro-F1:", round(max(scores), 4), "\nBest parameters:", best_params, f"\nSearch time: {bo_seconds:.0f}s")

## 10. Select the final model (training-CV evidence only)
The final model is the candidate with the highest **CV Macro-F1** among the five models and the tuned boosting model. The student-test results are not used for this decision.

In [ ]:
cv_scores = comparison["CV_Macro_F1"].to_dict()
cv_scores["Hist Gradient Boosting (Bayesian tuned)"] = tuned_row["CV_Macro_F1"]
final_name = max(cv_scores, key=cv_scores.get)
final_model = tuned if "tuned" in final_name else fitted[final_name]
final_scores = p_tuned if "tuned" in final_name else test_scores[final_name]
print("Final model:", final_name, "| CV Macro-F1 =", round(cv_scores[final_name], 4))